In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')

!pip install mne
import mne
import numpy as np
import pandas as pd
import os

In [ ]:
# ==== ERP 时间窗 & 阈值 ====
erp_windows = {
    'N1': (0.100, 0.160),
    'N280': (0.240, 0.300)
}
erp_threshold = -0.5  # instead of -3.0

# ==== Region electrode list ====
region_dict = {
    'Left Anterior':     ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20'],
    'Medial Anterior':   ['E12', 'E5', 'E19', 'E11', 'E4', 'E18', 'E16', 'E10', 'E15', 'E21', 'E14', 'E17', 'E127', 'E126'],
    'Right Anterior':    ['E117', 'E118', 'E124', 'E110', 'E109', 'E116', 'E123', 'E3', 'E9', 'E2', 'E122', 'E115', 'E114', 'E121', 'E1', 'E8', 'E125', 'E120', 'E113', 'E119'],
    'Left Central':      ['E29', 'E30', 'E41', 'E45', 'E46', 'E42', 'E47', 'E49', 'E37', 'E36'],
    'Medial Central':    ['E6', 'E7', 'E13', 'E31', 'E106', 'E112', 'E80', 'E54', 'E55', 'E79'],
    'Right Central':     ['E87', 'E93', 'E98', 'E102', 'E103', 'E107', 'E108', 'E104', 'E105', 'E111'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E53', 'E56', 'E57', 'E58', 'E59', 'E60', 'E63', 'E64', 'E65', 'E66', 'E68', 'E69', 'E70', 'E73'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77', 'E71', 'E76', 'E75', 'E74', 'E78', 'E82', 'E81'],
    'Right Posterior':   ['E83', 'E84', 'E85', 'E86', 'E88', 'E89', 'E90', 'E91', 'E92', 'E94', 'E95', 'E96', 'E97', 'E99', 'E100', 'E101']
}

# ==== evoked 文件目录 ====
data_root = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/Individual_Subjects/'

# ==== Subject 列表 ====
subject_dirs = [f for f in os.listdir(data_root) if f.startswith('sub')]
subject_ids = [s.split('-')[1] for s in subject_dirs]

# ==== 结果容器 ====
results = []

for sid in subject_ids:
    subj_path = os.path.join(data_root, f'sub-{sid}')

    for condition in ['Normal', 'Slowed']:
        fname = f'sub-{sid}_task-{condition}-ave.fif'
        fpath = os.path.join(subj_path, fname)

        try:
            evoked = mne.read_evokeds(fpath, condition=0)  # 默认第一个 evoked 对象

            for region, picks in region_dict.items():
                evoked_region = evoked.copy().pick_channels(picks)
                data = evoked_region.data
                times = evoked_region.times

                for component, (tmin, tmax) in erp_windows.items():
                    mask = (times >= tmin) & (times <= tmax)
                    mean_amp_uv = data[:, mask].mean() * 1e6  # Volt → μV
                    present = mean_amp_uv < erp_threshold

                    results.append({
                        'Subject': sid,
                        'Region': region,
                        'Condition': condition,
                        'ERP_Component': component,
                        'ERP_Present': present,
                        'Mean_Amplitude': mean_amp_uv
                    })

        except Exception as e:
            print(f"⚠️ Failed sub-{sid} {condition}: {e}")

# ==== 保存结果 ====
df_results = pd.DataFrame(results)
df_results
output_csv_path = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/ERP_Presence_Table.csv'
df_results.to_csv(output_csv_path, index=False)
print(f"✅ Saved to: {output_csv_path}")

# Bayesian Modelling Check

In [ ]:
from scipy.stats import norm
import pandas as pd

# ==== 模型参数 ====
t_star = 0.96  # 秒
delta = 0.05  # ERP 窗口宽度 ±50ms
theta = 0.6  # ERP 判断阈值（你原来的）
sigma_l = 0.03  # 声学似然的不确定性，固定

def compute_p_erp(mu_c, sigma_p, t_star=0.96, sigma_l=0.03, delta=0.05):
    sigma_post_sq = 1 / (1/sigma_p**2 + 1/sigma_l**2)
    mu_post = sigma_post_sq * (mu_c/sigma_p**2 + t_star/sigma_l**2)
    sigma_post = sigma_post_sq**0.5
    p_erp = norm(mu_post, sigma_post).cdf(t_star + delta) - norm(mu_post, sigma_post).cdf(t_star - delta)
    return p_erp

df_results = pd.read_csv('/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/ERP_Presence_Table.csv')

def get_mu_c(row):
    return 0.96 if row['Condition'] == 'Normal' else 1.15

def get_sigma_p(row):
    return 0.05  # 可换成每个 subject 拟合值

# 添加模型预测值
df_results['mu_c'] = df_results.apply(get_mu_c, axis=1)
df_results['sigma_p'] = df_results.apply(get_sigma_p, axis=1)
df_results['P_ERP'] = df_results.apply(
    lambda row: compute_p_erp(mu_c=row['mu_c'], sigma_p=row['sigma_p']), axis=1
)

# 二值化预测
df_results['ERP_Predict'] = df_results['P_ERP'] > theta

from sklearn.metrics import accuracy_score, confusion_matrix

acc = accuracy_score(df_results['ERP_Present'], df_results['ERP_Predict'])
print(f"Accuracy: {acc:.2f}")

print(confusion_matrix(df_results['ERP_Present'], df_results['ERP_Predict']))

from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

y_true = df_results['ERP_Present']
y_score = df_results['P_ERP']

fpr, tpr, _ = roc_curve(y_true, y_score)
auc = roc_auc_score(y_true, y_score)

plt.plot(fpr, tpr, label=f'AUC = {auc:.2f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Bayesian Model ERP Prediction")
plt.legend()
plt.show()


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import seaborn as sns
import os

# Load your CSV result file
df_results = pd.read_csv('/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/ERP_Presence_Table.csv')

# Example simulated z_t (replace with real z_t output from HDP-HMM)
# For demo purpose, let's assume K=5 hidden states
np.random.seed(42)
df_results['z_t'] = np.random.randint(0, 5, size=len(df_results))

# === Step 1: Compute state-dependent mu_c_k ===
K = df_results['z_t'].nunique()
mu_ck = {}

for cond in ['Normal', 'Neutral']:
    for k in range(K):
        mask = (df_results['Condition'] == cond) & (df_results['z_t'] == k)
        if mask.any():
            mu_ck[(cond, k)] = 0.96 if cond == 'Normal' else 1.15
        else:
            mu_ck[(cond, k)] = np.nan

# === Step 2: Compute mu_c(t) based on z_t ===
mu_c_list = []
for _, row in df_results.iterrows():
    cond = row['Condition']
    k = row['z_t']
    mu_c_list.append(mu_ck.get((cond, k), 1.0))  # fallback to 1.0 if not found

df_results['mu_c'] = mu_c_list
df_results['sigma_p'] = 0.05  # could be personalized later

# === Step 3: Compute posterior ERP probability P_ERP ===
t_star = 0.96
delta = 0.05
theta = 0.6
sigma_l = 0.03

def compute_p_erp(mu_c, sigma_p, t_star=0.96, sigma_l=0.03, delta=0.05):
    sigma_post_sq = 1 / (1/sigma_p**2 + 1/sigma_l**2)
    mu_post = sigma_post_sq * (mu_c/sigma_p**2 + t_star/sigma_l**2)
    sigma_post = sigma_post_sq**0.5
    return norm(mu_post, sigma_post).cdf(t_star + delta) - norm(mu_post, sigma_post).cdf(t_star - delta)

df_results['P_ERP'] = df_results.apply(
    lambda row: compute_p_erp(row['mu_c'], row['sigma_p']), axis=1
)
df_results['ERP_Predict'] = df_results['P_ERP'] > theta

# === Step 4: Evaluation ===
acc = accuracy_score(df_results['ERP_Present'], df_results['ERP_Predict'])
cm = confusion_matrix(df_results['ERP_Present'], df_results['ERP_Predict'])
auc = roc_auc_score(df_results['ERP_Present'], df_results['P_ERP'])
fpr, tpr, _ = roc_curve(df_results['ERP_Present'], df_results['P_ERP'])

# === Step 5: Plot ROC ===
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {auc:.2f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Sticky-HDP-HMM Modulated ERP Prediction")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
print(f"Accuracy: {acc:.3f}")
